In [ ]:
"""
LEGACY: TimescaleDB experiments (kept as-is, not used by the ingest)

Changes on 2026-10-06: the imports pointed to functions that no longer exist (read_csv_from_path, write_csv_to_path,
read_json_from_path, write_json_to_path); they now import the renamed functions from so.core.local_file_management.
The password is read from the environment variable SO_TIMESCALE_PASSWORD (default: the local Docker container's).
Requires the requirements file's sqlalchemy and psycopg2-binary, and a running TimescaleDB container (cell 2).
"""
None

In [18]:
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, MetaData, Table, text, Column, Integer, String, Float, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
# FILE MANIMULATION
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS OF THE RESEARCH WORKSPACE (THE OLD NAMES read_csv_from_path, ... NO LONGER EXIST)
import os
from so.core.local_file_management import read_csv_file_from_path, write_csv_file_to_path, read_json_file_from_path, write_json_file_to_path

In [19]:
'''
Run Container & Connect To Local Volume

docker run -d --name timescaledb_container -p 5432:5432 -e POSTGRES_USER=postgres -e POSTGRES_PASSWORD=mysecretpassword -e POSTGRES_DB=postgres -v "C:/Users/nico/Desktop/timescaledb_data:/var/lib/postgresql/data" timescale/timescaledb:latest-pg15

00cb42efb4ab76415d1317b3263890b6ecc04fffd3ee8a125c80e61a80346a33
'''
None

In [20]:
# DEFINE DATABASE CONNECTION SETTINGS
DB_NAME = "postgres"
DB_USER = "postgres"
# READ THE PASSWORD FROM THE ENVIRONMENT (DEFAULT: THE LOCAL DOCKER CONTAINER OF CELL 1)
DB_PASSWORD = os.environ.get("SO_TIMESCALE_PASSWORD", "mysecretpassword")
DB_HOST = "localhost"
DB_PORT = "5432"
# DEFINE A DATABASE URI
DB_URI = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
# CREATE A SQLALCHEMY ENGINE
engine = create_engine(DB_URI)

# FUNCTION: CONNECT TO DATABASE AND CHECK CONNECTION
def connect_to_db(alert_in=True):
    # TRY TO CONNECT
    try:
        # USE ENGINE TO CONNECT TO DATABASE
        with engine.connect() as conn:
            # DISPLAY INFORMATION
            print("✅ Timescale DB is: Connected") if alert_in==True else None
            # RETURN TRUE
            return True
    # CATCH EXCEPTION
    except Exception as e:
        # DISPLAY INFORMATION
        print(f"❌ Timescale DB is: Disconnected ({e})") if alert_in==True else None
        # RETURN FALSE
        return False

# CALL FUNCTION TO CONNECT TO DT
connected = connect_to_db()

✅ Timescale DB is: Connected


In [21]:
# FUNCTION: PREVIEW AVAILABLE TABLES
def preview_tables():
    # USE ENGINE TO CONNECT TO DATABASE
    with engine.connect() as conn:
        """
        Preview all tables in the database.
        """
        # QUERY STRING TO COLLECT ALL TABLE INFORMATION
        query = text("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        # COLLECT RESULT
        result = conn.execute(query)
        # SELECT TABLE NAME FROM RESULT
        table_list = [row[0] for row in result]
        # PREVIEW INFORMATION
        print("✅ Tables in the database:")
        # ITERATE OVER TABLES
        for table in table_list:
            # DISPLAY INFORMATION
            print(f" - {table}")
    # CLOSE ENGINE
    engine.dispose()

# CALL FUNCTION TO PREVIEW TABLES
preview_tables()

✅ Tables in the database:
 - stock_prices


In [ ]:
Base = declarative_base()

# FUNCTION: DYNAMICALLY CREATE TABLE IN DATABASE
def create_table(table_name, columns):
    """
    Dynamically create a table in the database with the specified table name and columns.

    :param table_name: Name of the table to create.
    :param columns: A dictionary of column names as keys and column types as values.
                    Example: {"symbol": String, "price": Float, "timestamp": DateTime}
    """
    # LOAD DATABASE METADATA
    metadata = MetaData()
    # LOAD INFORMATION ABOUT TABLE USING REFLECT
    metadata.reflect(bind=engine)
    # IF THE TABLE NAME EXISTS IN THE METADATA
    if table_name in metadata.tables:
        # DISPLAY INFORMATION
        print(f"❌ Table '{table_name}' already exists in the database.")
        # EXIT FUNCTION
        return
    
    # ENSURE THERE IS A PRIMARY KEY
    # IF ID IS NOT IN COLUMNS
    if "id" not in columns:
        # CREATE AN AUTO-INCREMENTED PRIMARY KEY IF NOT PROVIDED
        columns["id"] = Integer
    # DEFINE THE TABLE DYNAMICALLY
    table_columns = [
        Column(name, col_type, primary_key=(name == "id")) for name, col_type in columns.items()
    ]
    # CREATE A NEW CLASS THAT INHERITS FROM BASE
    dynamic_table = type(table_name, (Base,), {
        '__tablename__': table_name,
        '__table_args__': {'extend_existing': True},
        **{col.name: col for col in table_columns}
    })
    # CREATE THE TABLE IN THE DATABASE
    Base.metadata.create_all(engine)
    # DISPLAY INFORMATION
    print(f"✅ Table '{table_name}' created successfully with columns: {', '.join(columns.keys())}")

# DEFINE TABLE COLUMNS
# columns = {
#     "symbol": String(50),
#     "price": Float,
#     "volume": Integer,
#     "timestamp": DateTime,
# }
stock_columns = {
    "timestamp": DateTime,
    "open": Float,
    "high": Float,
    "low": Float,
    "close": Float,
    "volume": Integer,
    "created_at": DateTime,
}
# CREATE THE TABLE STOCK PRICES
create_table('SPY_table', stock_columns)
# create_table('PERSON', columns)
# create_table('stock_prices_table', columns)
# create_table('flowers_table', columns)

✅ Table 'SPY_table' created successfully with columns: timestamp, open, high, low, close, volume, created_at, id


: 

In [7]:
# FUNCTION: DYNAMICALLY DELETE TABLE NAME
def delete_table(table_name):
    """
    Dynamically delete (drop) a table from the database.

    :param table_name: Name of the table to delete.
    """
    # TRY TO DELETE TABLE
    try:
        # LOAD DATABASE METADATA
        metadata = MetaData()
        # LOAD INFORMATION ABOUT TABLE USING REFLECT
        metadata.reflect(bind=engine)
        # IF THE TABLE NAME EXISTS IN THE METADATA
        if table_name in metadata.tables:
            # COLLECT TABLE USING TABLE NAME
            table = Table(table_name, metadata, autoload_with=engine)
            # DROP TABLE
            table.drop(engine)
            # DISPLAY INFORMATION
            print(f"✅ Table '{table_name}' deleted successfully.")
        # IF THE TABLE NAME DOES NOT EXIST IN THE METADATA
        else:
            # TABLE DOES NOT EXIST
            print(f"⚠️ Table '{table_name}' does not exist.")
    # CATCH EXCEPTION
    except Exception as e:
        # DISPLAY INFORMATION
        print(f"❌ Error deleting table '{table_name}': {e}")

# Example usage:
# delete_table("stock_prices")
# delete_table("stock_prices_new")
# delete_table("person")
# delete_table("flowers_table")

In [8]:
# FUNCTION: INSERT MULTIPLE ROWS INTO A TABLE
def insert_multiple_rows(table_name, data):
    """
    Dynamically insert multiple rows into a given table by name.

    :param table_name: Name of the table to insert data into (string).
    :param data: A list of dictionaries containing row data.
                 Example: [{"symbol": "AAPL", "price": 150.5, "volume": 1000, "timestamp": datetime.now()}, ...]
    """
    # CREATE A SESSION
    Session = sessionmaker(bind=engine)
    # CREATE A SESSION
    session = Session()
    # COLLECT METADATA
    metadata = MetaData()
    # TRY TO INSERT DATA
    try:
        # REFLECT EXISTING TABLES IN THE DATABASE
        metadata.reflect(engine)
        # ENSURE THAT TABLE EXISTS
        if table_name not in metadata.tables:
            # RAISE ERROR
            raise ValueError(f"⚠️ Table '{table_name}' does not exist in the database.")
        # GET THE TABLE OBJECT DYNAMICALLY
        table = metadata.tables[table_name]
        # INSERT MULTIPLE ROWS INTO THE DATA
        session.execute(table.insert(), data)
        # COMMIT THE CHANGES
        session.commit()
        # DISPLAY INFORMATION
        print(f"✅ {len(data)} rows inserted successfully into '{table_name}'.")
    # CATCH EXCEPTION
    except Exception as e:
        # ROLLBACK THE SESSION
        session.rollback()
        # DISPLAY INFORMATION
        print(f"❌ Error inserting data into '{table_name}': {e}")
    # FINALLY
    finally:
        # CLOSE THE SESSION
        session.close()

# EXAMPLE DATA
stock_data = [
    {"symbol": "AAPL", "price": 150.5, "volume": 1000, "timestamp": datetime.now()},
    {"symbol": "GOOG", "price": 2800.0, "volume": 500, "timestamp": datetime.now()},
    {"symbol": "MSFT", "price": 299.7, "volume": 1200, "timestamp": datetime.now()},
    {"symbol": "XD", "price": 70, "volume": 500, "timestamp": datetime.now()}
]
# INSERT MULTIPLE ROWS 
# insert_multiple_rows('stock_prices', stock_data)

In [9]:
# FUNCTION: INSERT A SINGLE ROW INTO THE TABLE
def insert_new_stock_price(symbol, price, volume):
    """
    Inserts a new stock price into the stock_price table.
    """
    # CREATE A SESSION
    Session = sessionmaker(bind=engine)
    # CREATE A SESSION
    session = Session()
    # COLLECT METADATA
    metadata = MetaData()
    # REFLECT EXISTING TABLES IN THE DATABASE
    metadata.reflect(engine)
    # DYNAMICALLY FETCH THE TABLE
    table = metadata.tables.get("stock_prices")  # Dynamically fetch the table
    # IF THE TABLE DOES NOT EXIST
    if table is None:
        # DISPLAY INFORMATION
        print("❌ Table 'stock_prices' does not exist.")
        # EXIT FUNCTION
        return
    # DEFINE ROW
    new_row = {
        "symbol": symbol,
        "price": price,
        "volume": volume,
        "timestamp": datetime.now()
    }
    # TRY INSERT DATA
    try:
        # INSERT DATA
        session.execute(table.insert(), new_row)  # Insert data
        # COMMIT INSERTION
        session.commit()
        # DISPLAY INFORMATION
        print(f"✅ Inserted {symbol} at ${price} with volume {volume}")
    # CATCH EXCEPTION
    except Exception as e:
        # ROLLBACK THE SESSION
        session.rollback()
        # DISPLAY INFORMATION
        print(f"❌ Error inserting stock price: {e}")

# Example Usage
# insert_new_stock_price("AAPL", 150.75, 1200)
insert_new_stock_price("GOOGLE", 2801.50, 41)

✅ Inserted GOOGLE at $2801.5 with volume 41


In [10]:
# QUERY DATA USING SQLALCHEMY ENGINE
query = "SELECT * FROM stock_prices;"
# query = "SELECT * FROM stock_prices ORDER BY timestamp DESC;"
# query = "SELECT * FROM stock_prices ORDER BY timestamp DESC;"
# READ DATAFRAME
pdf = pd.read_sql(query, engine)
# PREVIEW DATAFRAME
pdf

,symbol,price,volume,timestamp,id
0,GOOGLE,2801.5,41,2025-02-04 16:19:49.802479,1
1,GOOGLE,2801.5,41,2025-02-04 16:19:50.500005,2
2,GOOGLE,2801.5,41,2025-02-04 16:19:51.277727,3
3,GOOGLE,2801.5,41,2025-02-04 16:19:52.020817,4
4,GOOGLE,2801.5,41,2025-02-04 16:19:53.107543,5
5,GOOGLE,2801.5,41,2025-02-04 16:22:14.725729,6
6,GOOGLE,2801.5,41,2025-02-04 16:25:11.748880,7
7,GOOGLE,2801.5,41,2025-02-04 16:37:32.972768,8
8,GOOGLE,2801.5,41,2025-02-15 19:19:32.181691,9


In [60]:
# # # SAVE CSV TO PATH
# write_csv_file_to_path(pdf, "C:/Users/nico/Desktop/stock_prices_backup.csv", "W")

In [10]:
# # READ THE CURRENT DATA FILE AS STRING VALUES
# read_pdf = read_csv_file_from_path( "stock_prices_backup.csv")
# # PREVIEW DATAFRAME
# read_pdf

In [11]:
# from pyspark.sql import SparkSession
# # ✅ Set JDBC Driver Path (Update with your actual path)
# JDBC_DRIVER_PATH = "C:/Program Files/Java/jdbc/postgresql-42.7.5.jar"  # <-- Update this path
# # ✅ Create SparkSession with PostgreSQL JDBC Driver
# spark = SparkSession.builder \
#     .appName("TimescaleDB_PySpark") \
#     .config("spark.jars", JDBC_DRIVER_PATH) \
#     .getOrCreate()

# # ✅ Define Database Properties
# db_properties = {
#     "user": "postgres",
#     "password": "mysecretpassword",
#     "driver": "org.postgresql.Driver"
# }
# # ✅ PostgreSQL JDBC URL
# db_url = "jdbc:postgresql://localhost:5432/postgres"
# # ✅ Read `stock_price` Table into a PySpark DataFrame
# sdf = spark.read.jdbc(url=db_url, table="stock_prices", properties=db_properties)
# # ✅ Show Data
# sdf.show()

In [12]:
# # DEFINE DATA FILE PATH
# data_file_path_str = "/Users/nico/Desktop/IMGM_data/rawzone/ibkr_data/ibkr_SP500_1m_20191230-20200103.csv"
# # READ THE CURRENT DATA FILE AS STRING VALUES
# read_pdf = read_csv_file_from_path(data_file_path_str)
# # PREVIEW DATAFRAME
# read_pdf